# Synthetic Tabular Data Generation - Staged Optimization Driver

**Enhanced notebook for synthetic tabular data generation with staged hyperparameter optimization.**

This notebook provides a staged optimization workflow with:
- Section 1: Setup and imports
- Section 2: Data loading, preprocessing, and EDA
- Section 3: Demo models with default parameters
- Section 4: **Staged Hyperparameter Optimization** (NEW)
  - 4.1: Configuration
  - 4.2: Pilot phase (15 trials) with time estimates
  - 4.3: Continuation options
  - 4.4: Diminishing returns analysis
  - 4.5: Additional batches (optional)
- Section 5: Final models with best parameters

Based on STG-Driver-breast-cancer.ipynb with staged optimization enhancements.

## 1 Setup and Configuration

In [1]:
# Code Chunk ID: CHUNK_1_0_0_A - Import Setup Module
# Import all functionality from setup.py
import sys
sys.path.insert(0, "/home/ec2-user/SageMaker/tableGenCompare")

from setup import *

# Refresh session timestamp to current date
refresh_session_timestamp()

# ============================================================================
# FRESH START CONTROL - Set to True to wipe all checkpoints and start over
# ============================================================================
FRESH_START = True   # <-- Change to True to clear ALL saved progress
# ============================================================================

print("SETUP MODULE IMPORTED SUCCESSFULLY!")
print(f"FRESH_START = {FRESH_START}")
print("=" * 60)

[OK] Essential data science libraries imported successfully!
[OK] Model registry loaded from src/models/registry.py
[OK] Batch training module loaded from src/models/batch_training.py
[OK] Search spaces module loaded from src/models/search_spaces.py
[OK] Batch optimization module loaded from src/models/batch_optimization.py
[OK] Staged optimization module loaded from src/models/staged_optimization.py
[CONFIG] Session timestamp: 2026-03-26
[OK] Parameter management functions loaded from src/utils/parameters.py
[OK] Hyperparameter optimization evaluation functions loaded from src/evaluation/hyperparameters.py
[OK] Optuna objective functions for PATE-GAN and MEDGAN loaded (Phase 5)
Detected sklearn 1.7.2 - applying compatibility patch...
INFO: Applying sklearn compatibility patches for version 1.7.2
Global sklearn compatibility patch applied successfully
CTAB-GAN imported successfully
[OK] CTAB-GAN+ detected and available
[OK] GANerAidModel imported successfully from src.models.implementa

## 2 Data Loading and Pre-processing

### 2.1 Configuration

**USER ATTENTION NEEDED**: Edit the `NOTEBOOK_CONFIG` dictionary below to match your dataset.

In [2]:
# Code Chunk ID: CHUNK_2_1_1_A - NOTEBOOK CONFIGURATION
# ============================================================================
# USER CONFIGURATION - Edit ONLY this block for your dataset
# ============================================================================

NOTEBOOK_CONFIG = {
    # ========== REQUIRED: Dataset Settings ==========
    "data_file": "data/Pakistani_Diabetes_Dataset.csv",  # Path to your CSV file
    "target_column": "Outcome",                 # Target/outcome column name

    # ========== OPTIONAL: Dataset Metadata ==========
    "dataset_name": "Pakistani Diabetes Dataset",      # Display name
    "dataset_identifier_override": None,          # Override auto-detected ID (or None)

    # ========== OPTIONAL: Column Configuration ==========
    # Binary 0/1 flags in this dataset (7 features):
    "categorical_columns": ["Gender", "Rgn ", "his", "vision", "dipsia", "uria", "neph"],
    "task_type": "classification",

    # ========== OPTIONAL: Fairness Evaluation ==========
    # Protected attribute for fairness metrics (use cleaned column name after preprocessing).
    # Gender is binary (0/1) and survives preprocessing as-is.
    "protected_col": "gender",

    # ========== OPTIONAL: Data Subsetting ==========
    "use_row_subset": False,                      # 912 rows - small enough to use all
    "sample_n": 500,                              # Number of rows to sample
    "sample_random_state": 42,                    # Random seed for reproducibility

    # ========== OPTIONAL: Missing Data Handling ==========
    "missing_strategy": "none",                   # No missing values in this dataset
    "mice_max_iter": 10,                          # Max iterations for MICE imputation

    # ========== OPTIONAL: Model Selection ==========
    "models_to_run": ["ctgan", "tvae", "copulagan", "ctabgan", "ctabganplus", "pategan", "medgan"],     # "all" or list like ["ctgan", "tvae"]
    # "models_to_run": ["ctgan", "tvae", "copulagan", "ctabgan", "ctabganplus", "ganeraid", "pategan", "medgan"],  

    # ========== OPTIONAL: Tuning Configuration ==========
    "tuning_mode": "full",                        # "smoke" (quick) | "full" (thorough)
    "n_trials_smoke": 15,                          # Trials for smoke testing / pilot phase
}

print("NOTEBOOK_CONFIG set successfully!")
print(f"  Data file: {NOTEBOOK_CONFIG['data_file']}")
print(f"  Target column: {NOTEBOOK_CONFIG['target_column']}")
print(f"  Protected column: {NOTEBOOK_CONFIG.get('protected_col', None)}")
print(f"  Models to run: {NOTEBOOK_CONFIG['models_to_run']}")
print(f"  Tuning mode: {NOTEBOOK_CONFIG['tuning_mode']}")

NOTEBOOK_CONFIG set successfully!
  Data file: data/Pakistani_Diabetes_Dataset.csv
  Target column: Outcome
  Protected column: gender
  Models to run: ['ctgan', 'tvae', 'copulagan', 'ctabgan', 'ctabganplus', 'pategan', 'medgan']
  Tuning mode: full


### 2.2 Load and Preprocess Data

In [3]:
# Code Chunk ID: CHUNK_2_1_2_A - LOAD AND PREPROCESS DATA
# ============================================================================
# This uses the config-driven preprocessing pipeline
# ============================================================================

if not FRESH_START and 'checkpoint' in dir() and hasattr(checkpoint, 'exists') and checkpoint.exists('section_2'):
    saved = checkpoint.load('section_2')
    data = saved['data']
    original_data = saved['original_data']
    target_column = saved['target_column']
    DATASET_IDENTIFIER = saved['DATASET_IDENTIFIER']
    categorical_columns = saved['categorical_columns']
    metadata = saved['metadata']
    models_to_run = saved['models_to_run']
    RUN_MODE = saved['RUN_MODE']
    TARGET_COLUMN = target_column
    print("[RESUME] Loaded Section 2 from checkpoint")
else:
    # Load and preprocess data using the config
    (
        data,                  # Processed DataFrame
        original_data,         # Copy for reference
        target_column,         # Target column name (cleaned)
        DATASET_IDENTIFIER,    # Dataset identifier for results paths
        categorical_columns,   # List of categorical columns
        metadata               # Full preprocessing metadata
    ) = load_and_preprocess_from_config(NOTEBOOK_CONFIG)

    # Set aliases for backward compatibility with existing notebook cells
    TARGET_COLUMN = target_column

    # Get models to run based on config
    models_to_run = get_models_to_run(NOTEBOOK_CONFIG)

    # Set RUN_MODE for backward compatibility with Section 4 tuning cells
    RUN_MODE = "debug" if NOTEBOOK_CONFIG['tuning_mode'] == "smoke" else "full"

# Initialize checkpoint system (now that DATASET_IDENTIFIER is known)
checkpoint = SectionCheckpoint(DATASET_IDENTIFIER)

# If FRESH_START, wipe all checkpoints and optimization studies
if FRESH_START:
    n_removed = checkpoint.clear_all()
    print(f"[FRESH START] Cleared {n_removed} checkpoint(s) and optimization studies")

existing = checkpoint.list_checkpoints()
if existing:
    print(f"[CHECKPOINT] Found {len(existing)} existing checkpoint(s): {existing}")

# Save Section 2 checkpoint (idempotent - overwrites if exists)
if not checkpoint.exists('section_2'):
    checkpoint.save('section_2', {
        'data': data, 'original_data': original_data,
        'target_column': target_column, 'DATASET_IDENTIFIER': DATASET_IDENTIFIER,
        'categorical_columns': categorical_columns, 'metadata': metadata,
        'models_to_run': models_to_run, 'RUN_MODE': RUN_MODE,
    })

print()
print("=" * 60)
print("PREPROCESSING COMPLETE")
print("=" * 60)
print(f"  Dataset identifier: {DATASET_IDENTIFIER}")
print(f"  Processed shape: {data.shape}")
print(f"  Target column: {target_column}")
print(f"  Task type: {metadata['task_type']}")
print(f"  Categorical columns: {categorical_columns}")
print(f"  Models to run: {models_to_run}")
print(f"  RUN_MODE: {RUN_MODE}")
print(f"  Session: {SESSION_TIMESTAMP}")
print(f"  Results path: {get_results_path(DATASET_IDENTIFIER, 2)}")
print("=" * 60)

[LOAD] Loading data from: data/Pakistani_Diabetes_Dataset.csv
[LOAD] Loaded 912 rows, 19 columns
[PREPROCESS] Starting preprocessing pipeline
[PREPROCESS] Input shape: (912, 19)
[PREPROCESS] Categorical columns: ['gender', 'rgn', 'his', 'vision', 'dipsia', 'uria', 'neph']
[PREPROCESS] Task type: classification
[PREPROCESS] Final shape: (912, 19)
[PREPROCESS] Dataset identifier: pakistani-diabetes-dataset
[FLUSH] No previous studies found in results/pakistani-diabetes-dataset/optimization_studies — starting clean
[FRESH START] Cleared 0 checkpoint(s) and optimization studies

PREPROCESSING COMPLETE
  Dataset identifier: pakistani-diabetes-dataset
  Processed shape: (912, 19)
  Target column: outcome
  Task type: classification
  Categorical columns: ['gender', 'rgn', 'his', 'vision', 'dipsia', 'uria', 'neph']
  Models to run: ['ctgan', 'tvae', 'copulagan', 'ctabgan', 'ctabganplus', 'pategan', 'medgan']
  RUN_MODE: full
  Session: 2026-03-26
  Results path: results/pakistani-diabetes-dat

### 2.3 Exploratory Data Analysis (EDA)

Comprehensive EDA with automated file export to results directory.

In [4]:
# Code Chunk ID: CHUNK_2_1_EDA - COMPREHENSIVE EDA
# ============================================================================
# Run comprehensive EDA with single function call
# ============================================================================

from src.data.eda import run_comprehensive_eda

eda_results = run_comprehensive_eda(
    data=data,
    target_column=target_column,
    dataset_identifier=DATASET_IDENTIFIER,
    dataset_name=NOTEBOOK_CONFIG.get('dataset_name'),
    categorical_columns=categorical_columns,
    verbose=True
)

# Update categorical_columns from EDA results (in case auto-detected)
categorical_columns = eda_results['categorical_columns']

print(f"\nEDA Results Summary:")
print(f"  Files generated: {len(eda_results['files_generated'])}")
print(f"  Categorical columns: {categorical_columns}")
print(f"  Balance ratio: {eda_results.get('balance_ratio', 'N/A')}")

COMPREHENSIVE EXPLORATORY DATA ANALYSIS
Dataset: Pakistani Diabetes Dataset
Target column: outcome
Results path: results/pakistani-diabetes-dataset/2026-03-26/Section-2

[1/6] Dataset Overview...
   Dataset Name.................. Pakistani Diabetes Dataset
   Shape......................... 912 rows x 19 columns
   Memory Usage.................. 0.13 MB
   Total Missing Values.......... 0
   Missing Percentage............ 0.00%
   Duplicate Rows................ 2
   Numeric Columns............... 19
   Categorical Columns........... 0

[2/6] Column Analysis...
   Saved: column_analysis.csv (19 columns)

[3/6] Target Variable Analysis...
   Saved: target_analysis.csv
   Saved: target_balance_metrics.csv
   Balance Ratio: 0.877 (Balanced)

[4/6] Feature Distributions...
   Saved: 3 distribution file(s) (18 features)

[5/6] Correlation Analysis...
   Saved: mixed_association_heatmap.png
   Saved: association_matrix.csv
   Saved: target_correlations.csv (18 features)

[6/6] Configuration Va

## 3 Demo All Models with Default Parameters

### 3.1 Batch Model Training

In [5]:
# Code Chunk ID: CHUNK_3_1_BATCH
# ============================================================================
# SECTION 3.1 - BATCH MODEL TRAINING
# Train all models configured in NOTEBOOK_CONFIG['models_to_run']
# ============================================================================

from src.models.batch_training import train_models_batch, extract_synthetic_data_to_globals

# Run batch training for all configured models (checkpoint resumes completed models)
training_results = train_models_batch(
    data=data,
    models_to_run=models_to_run,
    target_column=target_column,
    categorical_columns=categorical_columns,
    n_samples=len(data),
    random_state=42,
    verbose=True,
    checkpoint=checkpoint
)

# Extract synthetic_data_* variables for Section 3.2 compatibility
created_vars = extract_synthetic_data_to_globals(training_results, globals())
print(f"\nCreated variables: {created_vars}")

# Also create model_* variables for backward compatibility
for model_name, result in training_results.items():
    if result['status'] == 'success' and result.get('model') is not None:
        globals()[f'{model_name}_model'] = result['model']


BATCH MODEL TRAINING
Models to train: 7
Dataset shape: (912, 19)
Target column: outcome
Samples to generate: 912
GPU available: NVIDIA A10G (22.1 GB)
Device assignments:
  - CTGAN: cuda
  - TVAE: cuda
  - CopulaGAN: cuda
  - CTABGAN: cuda
  - CTABGAN+: cuda
  - PATE-GAN: cuda
  - MEDGAN: cuda


[1/7] Training CTGAN...
--------------------------------------------------
  Device: cuda
  Training CTGAN...


Gen. (-1.06) | Discrim. (-0.05): 100%|██████████| 300/300 [00:09<00:00, 32.27it/s]


  Generating 912 synthetic samples...
  [OK] CTGAN completed in 15.61s
  Synthetic data shape: (912, 19)

[2/7] Training TVAE...
--------------------------------------------------
  Device: cuda
  Training TVAE...
  Generating 912 synthetic samples...
[OK] Target integrity functions loaded from src/data/target_integrity.py
  [OK] TVAE completed in 10.62s
  Synthetic data shape: (912, 19)

[3/7] Training CopulaGAN...
--------------------------------------------------
  Device: cuda
  Training CopulaGAN...
  Generating 912 synthetic samples...
  [OK] CopulaGAN completed in 12.17s
  Synthetic data shape: (912, 19)

[4/7] Training CTABGAN...
--------------------------------------------------
  Device: cuda
  Training CTABGAN...


100%|██████████| 300/300 [00:26<00:00, 11.44it/s]


Finished training in 27.802804946899414  seconds.
  Generating 912 synthetic samples...
  [OK] CTABGAN completed in 27.88s
  Synthetic data shape: (912, 19)

[5/7] Training CTABGAN+...
--------------------------------------------------
  Device: cuda
  Training CTABGAN+...


100%|██████████| 400/400 [00:47<00:00,  8.44it/s]


Finished training in 49.107192516326904  seconds.
  Generating 912 synthetic samples...
  [OK] CTABGAN+ completed in 49.18s
  Synthetic data shape: (912, 19)

[6/7] Training PATE-GAN...
--------------------------------------------------
  Device: cuda
  Training PATE-GAN...
  Generating 912 synthetic samples...
  [OK] PATE-GAN completed in 9.90s
  Synthetic data shape: (912, 19)

[7/7] Training MEDGAN...
--------------------------------------------------
  Device: cuda
  Training MEDGAN...
  Generating 912 synthetic samples...
  [OK] MEDGAN completed in 13.21s
  Synthetic data shape: (912, 19)

BATCH TRAINING SUMMARY
Total models: 7
Successful: 7
Failed: 0

Successful models:
  - CTGAN: 15.61s
  - TVAE: 10.62s
  - CopulaGAN: 12.17s
  - CTABGAN: 27.88s
  - CTABGAN+: 49.18s
  - PATE-GAN: 9.90s
  - MEDGAN: 13.21s

Created variables: ['synthetic_data_ctgan', 'synthetic_data_tvae', 'synthetic_data_copulagan', 'synthetic_data_ctabgan', 'synthetic_data_ctabganplus', 'synthetic_data_pategan', 

### 3.2 Batch Evaluation

In [6]:
# Code Chunk ID: CHUNK_3_2_0_A
# ============================================================================
# SECTION 3.2 - BATCH EVALUATION FOR ALL TRAINED MODELS
# ============================================================================

print("SECTION 3.2 - COMPREHENSIVE BATCH EVALUATION")
print("=" * 60)

if checkpoint.exists('section_3.2'):
    section3_results = checkpoint.load('section_3.2')['results']
    print("[RESUME] Loaded Section 3.2 evaluation from checkpoint")
else:
    section3_results = evaluate_all_available_models(
        section_number=3,
        scope=globals(),
        models_to_evaluate=None,
        real_data=None,
        target_col=None,
        protected_col=NOTEBOOK_CONFIG.get("protected_col")
    )
    checkpoint.save('section_3.2', {'results': section3_results})

if section3_results:
    print(f"\nSECTION 3 BATCH EVALUATION COMPLETED!")
    print(f"Evaluated {len(section3_results)} models successfully")
    
    # Show ranking by quality score
    best_models = []
    for model_name, results in section3_results.items():
        if 'error' not in results:
            quality_score = results.get('overall_quality_score', 0)
            best_models.append((model_name, quality_score))
    
    if best_models:
        best_models.sort(key=lambda x: x[1], reverse=True)
        print(f"\nRANKING BY QUALITY SCORE:")
        for i, (model, score) in enumerate(best_models, 1):
            print(f"   {i}. {model}: {score:.3f}")
else:
    print("\nNo models available for evaluation")

SECTION 3.2 - COMPREHENSIVE BATCH EVALUATION
[OK] Batch evaluation functions loaded from src/evaluation/batch.py
[SEARCH] UNIFIED BATCH EVALUATION - SECTION 3
[INFO] Dataset: pakistani-diabetes-dataset
[INFO] Target column: outcome
[INFO] Protected column: gender
[INFO] MIA evaluation: OFF
[INFO] Variable pattern: standard
[INFO] Found 7 trained models:
   [OK] CTGAN
   [OK] CTABGAN
   [OK] CTABGANPLUS
   [OK] CopulaGAN
   [OK] TVAE
   [OK] MEDGAN
   [OK] PATEGAN

==================== EVALUATING CTGAN ====================
[SEARCH] CTGAN - COMPREHENSIVE DATA QUALITY EVALUATION
[FOLDER] Output directory: results/pakistani-diabetes-dataset/2026-03-26/Section-3/CTGAN

[1] STATISTICAL SIMILARITY
------------------------------
   [CHART] Average Statistical Similarity: 0.834

[2] PCA COMPARISON ANALYSIS WITH OUTCOME COLOR-CODING
--------------------------------------------------
   [CHART] PCA comparison plot saved: pca_comparison_with_outcome.png
   [CHART] PCA Overall Similarity: 0.017
   

## 4 Staged Hyperparameter Optimization

This section uses a staged approach to hyperparameter optimization:

- **Smoke mode** (`tuning_mode="smoke"`): Runs 10 pilot trials per model, then displays
  time-budget recommendations (how many trials fit in 1 hour, how long 20 trials would take).
  Section 5 uses the pilot results directly.
- **Full mode** (`tuning_mode="full"`): Runs a pilot phase, displays time estimates, then
  presents three continuation strategies in cell 4.3.  The user reviews the estimates and
  **uncomments one option** before running the cell.

### 4.1 Configuration

Create the `StagedOptimizationManager`.  `TUNING_MODE` and `PILOT_TRIALS` are derived
from `NOTEBOOK_CONFIG` so the same code works for both smoke and full runs.

In [7]:
# Code Chunk ID: CHUNK_4_1_CONFIG
# ============================================================================
# SECTION 4.1 - STAGED OPTIMIZATION CONFIGURATION
# ============================================================================

from src.models.staged_optimization import (
    StagedOptimizationManager,
    StagedOptimizationConfig,
    flush_previous_runs
)

# Flush optimization studies if FRESH_START is set
if FRESH_START:
    flush_previous_runs(DATASET_IDENTIFIER)

# Derive tuning mode and pilot size from NOTEBOOK_CONFIG
TUNING_MODE = NOTEBOOK_CONFIG.get("tuning_mode", "smoke")
PILOT_TRIALS = 10 if TUNING_MODE == "smoke" else NOTEBOOK_CONFIG.get("n_trials_smoke", 10)

# Configure staged optimization
staged_config = StagedOptimizationConfig(
    pilot_trials=PILOT_TRIALS,
    verbose_level=1,           # 0=silent, 1=concise, 2=detailed
    persistence_dir=f"results/{DATASET_IDENTIFIER}/optimization_studies",
    run_mode=RUN_MODE,         # "debug" or "full"
    random_state=42,
    continue_on_error=True
)

# Create the manager
manager = StagedOptimizationManager(
    data=data,
    target_column=target_column,
    categorical_columns=categorical_columns,
    dataset_identifier=DATASET_IDENTIFIER,
    config=staged_config
)

print("Staged Optimization Manager created!")
print(f"  Tuning mode: {TUNING_MODE}")
print(f"  Pilot trials: {staged_config.pilot_trials}")
print(f"  Run mode: {staged_config.run_mode}")
print(f"  Persistence dir: {staged_config.persistence_dir}")
print(f"  FRESH_START: {FRESH_START}")

[FLUSH] No previous studies found in results/pakistani-diabetes-dataset/optimization_studies — starting clean
Staged Optimization Manager created!
  Tuning mode: full
  Pilot trials: 15
  Run mode: full
  Persistence dir: results/pakistani-diabetes-dataset/optimization_studies
  FRESH_START: True


### 4.2 Run Pilot Phase

Run a pilot phase to establish time estimates.

- **Smoke mode**: 10 trials + smoke recommendations table (trials in 1 hr, time for 20 trials).
- **Full mode**: Pilot trials + time estimates for planning continuation.

In [8]:
# Code Chunk ID: CHUNK_4_2_PILOT
# ============================================================================
# SECTION 4.2 - RUN PILOT PHASE
# ============================================================================

# Run pilot phase (uses PILOT_TRIALS from cell 4.1)
pilot_states = manager.run_pilot(
    models_to_run=models_to_run
)

# Display time estimates
print("\n" + "="*60)
print("PILOT PHASE COMPLETE - TIME ESTIMATES")
print("="*60)

time_estimates = manager.get_time_estimates()
if len(time_estimates) > 0:
    print(time_estimates.to_string(index=False))
else:
    print("No time estimates available")

# Show best scores so far
print("\nBest scores after pilot:")
for model_name, state in pilot_states.items():
    print(f"  {model_name}: {state.best_score:.4f} ({state.total_trials} trials)")

# Smoke mode: show budget recommendations
if TUNING_MODE == "smoke":
    print("\n" + "="*60)
    print("SMOKE MODE - TIME BUDGET RECOMMENDATIONS")
    print("="*60)
    smoke_recs = manager.get_smoke_recommendations()
    print(smoke_recs.to_string(index=False))
    print("\nTo run a full optimization, set tuning_mode='full' in NOTEBOOK_CONFIG")
    print("and use the recommended_pilot column to guide n_trials_smoke.")

[I 2026-03-26 19:02:11,429] A new study created in memory with name: ctgan_hpo_pakistani-diabetes-dataset
Trial failed for ctgan: No trials are completed yet.



STAGED OPTIMIZATION - PILOT PHASE
Models to optimize: 7
Pilot trials per model: 15
Dataset identifier: pakistani-diabetes-dataset


[PILOT] Optimizing CTGAN...
--------------------------------------------------
  [ERROR] CTGAN failed: No trials are completed yet.

[PILOT] Optimizing TVAE...
--------------------------------------------------
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.6047
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9452
[CHART] Combined Score: 0.7409 (Similarity: 0.6047, Accuracy: 0.9452)
[tvae] Trial 1: Combined Score: 0.7409 (Similarity: 0.6047, Accuracy: 0.9452) Best Score so far: 0.7409
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.6660
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9688
[CHART] Combined Score: 0.7871 (Similarity: 0.6660, Accuracy: 0.9688)
[tvae] Trial 2: Combined Score: 0.7871 (Similari

100%|██████████| 250/250 [00:27<00:00,  9.11it/s]


Finished training in 29.88653063774109  seconds.
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5977
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9666
[CHART] Combined Score: 0.7452 (Similarity: 0.5977, Accuracy: 0.9666)
[ctabgan] Trial 1: Combined Score: 0.7452 (Similarity: 0.5977, Accuracy: 0.9666) Best Score so far: 0.7452


100%|██████████| 200/200 [00:21<00:00,  9.51it/s]


...

Finished training in 379.69154834747314  seconds.
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5724
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9720
[CHART] Combined Score: 0.7322 (Similarity: 0.5724, Accuracy: 0.9720)
[ctabganplus] Trial 13: Combined Score: 0.7322 (Similarity: 0.5724, Accuracy: 0.9720) Best Score so far: 0.7809


100%|██████████| 350/350 [00:36<00:00,  9.69it/s]


Finished training in 39.33088231086731  seconds.
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5770
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9731
[CHART] Combined Score: 0.7355 (Similarity: 0.5770, Accuracy: 0.9731)
[ctabganplus] Trial 14: Combined Score: 0.7355 (Similarity: 0.5770, Accuracy: 0.9731) Best Score so far: 0.7809


100%|██████████| 850/850 [09:17<00:00,  1.52it/s]


Finished training in 560.0127191543579  seconds.
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.6067
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9797
[CHART] Combined Score: 0.7559 (Similarity: 0.6067, Accuracy: 0.9797)
[ctabganplus] Trial 15: Combined Score: 0.7559 (Similarity: 0.6067, Accuracy: 0.9797) Best Score so far: 0.7809
  [OK] CTABGAN+: 15 trials in 3395.2s
  Best score: 0.7809

[PILOT] Optimizing PATE-GAN...
--------------------------------------------------
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.3082
[OK] TRTS Evaluation: 2 scenarios, Average: 0.2336
[CHART] Combined Score: 0.2783 (Similarity: 0.3082, Accuracy: 0.2336)
[pategan] Trial 1: Combined Score: 0.2783 (Similarity: 0.3082, Accuracy: 0.2336) Best Score so far: 0.2783
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity An

In [9]:
# ============================================================================
# DIMINISHING RETURNS ANALYSIS
# ============================================================================
print("DIMINISHING RETURNS ANALYSIS")
print("="*60)

convergence_report = manager.get_diminishing_returns_report()

if len(convergence_report) > 0:
    print(convergence_report.to_string(index=False))
    
    print("\nInterpretation:")
    print("-" * 40)
    for _, row in convergence_report.iterrows():
        print(f"  {row['model']}: {row['recommendation']}")
        if row['has_plateaued']:
            print(f"    -> Model has plateaued at score {row['best_score']:.4f}")
        else:
            print(f"    -> Improvement rate: {row['improvement_rate']:.4f}")
else:
    print("No convergence data available")

DIMINISHING RETURNS ANALYSIS
      model  trials  best_score  improvement_rate  total_improvement  has_plateaued                         recommendation
       tvae      15    0.800137          0.010808           0.059257          False Consider stopping - minor improvements
  copulagan      15    0.754877          0.000000           0.044995           True                 Stop - plateau reached
    ctabgan      15    0.749243          0.000000           0.004028           True                 Stop - plateau reached
ctabganplus      15    0.780918          0.024423           0.049622          False Consider stopping - minor improvements
    pategan      15    0.429096          0.155879           0.150773          False             Continue - still improving
     medgan      15    0.516175          0.000000           0.166682           True                 Stop - plateau reached

Interpretation:
----------------------------------------
  tvae: Consider stopping - minor improvements
    -

### 4.3 Continuation (Full Mode Only)

**Full mode only** - Review the pilot results and time estimates above, then
uncomment **one** of the three options below and adjust the values before running.

- **Option (i)**: Common trial count for all models
- **Option (ii)**: Per-model trial counts
- **Option (iii)**: Time-based budget (minutes per model)

Models not in `models_to_run` are silently ignored, so listing all 8 is safe.

In [10]:
# Code Chunk ID: CHUNK_4_3_CONTINUE
# ============================================================================
# SECTION 4.3 - CONTINUATION (Full mode only - choose ONE option)
# ============================================================================

if TUNING_MODE == "smoke":
    print("[SMOKE MODE] Skipping continuation - using pilot results for Section 5.")

else:
    # Choose ONE option below, then run this cell.

    # OPTION (i): Common trials for all models
    # continuation_states = manager.continue_optimization(additional_trials=30)

    # OPTION (ii): Per-model trials - adjust counts per model
    continuation_states = manager.continue_optimization(
        trials_per_model={
            "ctgan": 30,
            # "tvae": 30,
            # "copulagan": 30,
            # "ctabgan": 30,
            "ctabganplus": 15,
            # "ganeraid": 30,
            # "pategan": 30,
            "medgan": 50,
        }
    )

    # OPTION (iii): Time-based budget - minutes per model
    # continuation_states = manager.continue_optimization(
    #     time_budget_minutes={
    #         "ctgan": 60,
    #         "tvae": 60,
    #         "copulagan": 60,
    #         "ctabgan": 60,
    #         "ctabganplus": 60,
    #         "ganeraid": 60,
    #         "pategan": 60,
    #         "medgan": 60,
    #     }
    # )

    print("[FULL MODE] Continuation completed.")


STAGED OPTIMIZATION - CONTINUATION PHASE
  ctgan: 30 additional trials
  ctabganplus: 15 additional trials
  medgan: 50 additional trials


[CONTINUE] Optimizing CTGAN...
--------------------------------------------------


Gen. (-0.97) | Discrim. (-0.32): 100%|██████████| 450/450 [01:04<00:00,  6.95it/s]


[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5349
[OK] TRTS Evaluation: 2 scenarios, Average: 0.5740
[CHART] Combined Score: 0.5506 (Similarity: 0.5349, Accuracy: 0.5740)
[ctgan] Trial 1: Combined Score: 0.5506 (Similarity: 0.5349, Accuracy: 0.5740) Best Score so far: 0.5506


Gen. (-0.22) | Discrim. (-0.45): 100%|██████████| 750/750 [01:52<00:00,  6.67it/s]


[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5381
[OK] TRTS Evaluation: 2 scenarios, Average: 0.7473
[CHART] Combined Score: 0.6217 (Similarity: 0.5381, Accuracy: 0.7473)
[ctgan] Trial 2: Combined Score: 0.6217 (Similarity: 0.5381, Accuracy: 0.7473) Best Score so far: 0.6217
[ctgan] Trial 3: Combined Score: 0.0000 (Similarity: 0.0000, Accuracy: 0.0000) Best Score so far: 0.6217
[ctgan] Trial 4: Combined Score: 0.0000 (Similarity: 0.0000, Accuracy: 0.0000) Best Score so far: 0.6217


...

Finished training in 535.0118854045868  seconds.
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.6205
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9792
[CHART] Combined Score: 0.7640 (Similarity: 0.6205, Accuracy: 0.9792)
[ctabganplus] Trial 28: Combined Score: 0.7640 (Similarity: 0.6205, Accuracy: 0.9792) Best Score so far: 0.7809


100%|██████████| 950/950 [11:09<00:00,  1.42it/s]


Finished training in 671.4340744018555  seconds.
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.6622
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9797
[CHART] Combined Score: 0.7892 (Similarity: 0.6622, Accuracy: 0.9797)
[ctabganplus] Trial 29: Combined Score: 0.7892 (Similarity: 0.6622, Accuracy: 0.9797) Best Score so far: 0.7892


100%|██████████| 950/950 [02:31<00:00,  6.27it/s]


Finished training in 153.6345729827881  seconds.
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.6048
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9748
[CHART] Combined Score: 0.7528 (Similarity: 0.6048, Accuracy: 0.9748)
[ctabganplus] Trial 30: Combined Score: 0.7528 (Similarity: 0.6048, Accuracy: 0.9748) Best Score so far: 0.7892
  [OK] CTABGAN+: 15 trials in 7436.5s
  Best score: 0.7892

[CONTINUE] Optimizing MEDGAN...
--------------------------------------------------
  Resuming from 15 existing trials
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.4093
[OK] TRTS Evaluation: 2 scenarios, Average: 0.3657
[CHART] Combined Score: 0.3919 (Similarity: 0.4093, Accuracy: 0.3657)
[medgan] Trial 16: Combined Score: 0.3919 (Similarity: 0.4093, Accuracy: 0.3657) Best Score so far: 0.5162
[TARGET] Enhanced objective function using target 

In [11]:
# ============================================================================
# DIMINISHING RETURNS ANALYSIS (post-continuation)
# ============================================================================

if TUNING_MODE == "full":
    print("DIMINISHING RETURNS ANALYSIS")
    print("="*60)

    convergence_report = manager.get_diminishing_returns_report()

    if len(convergence_report) > 0:
        print(convergence_report.to_string(index=False))

        print("\nInterpretation:")
        print("-" * 40)
        for _, row in convergence_report.iterrows():
            print(f"  {row['model']}: {row['recommendation']}")
            if row['has_plateaued']:
                print(f"    -> Model has plateaued at score {row['best_score']:.4f}")
            else:
                print(f"    -> Improvement rate: {row['improvement_rate']:.4f}")
    else:
        print("No convergence data available")
else:
    print("[SMOKE MODE] Skipping post-continuation analysis.")

DIMINISHING RETURNS ANALYSIS
      model  trials  best_score  improvement_rate  total_improvement  has_plateaued                         recommendation
      ctgan      30    0.694665          0.000000           0.144090           True                 Stop - plateau reached
       tvae      15    0.800137          0.010808           0.059257          False Consider stopping - minor improvements
  copulagan      15    0.754877          0.000000           0.044995           True                 Stop - plateau reached
    ctabgan      15    0.749243          0.000000           0.004028           True                 Stop - plateau reached
ctabganplus      30    0.789216          0.010515           0.057921          False Consider stopping - minor improvements
    pategan      15    0.429096          0.155879           0.150773          False             Continue - still improving
     medgan      65    0.517247          0.000000           0.167753           True                 Stop - pla

### 4.5 Additional Batches (Full Mode, Optional)

If the diminishing returns analysis suggests continuing, uncomment one option below.
You can re-run this cell multiple times.

In [ ]:
# Code Chunk ID: CHUNK_4_5_ADDITIONAL

# ============================================================================
# SECTION 4.5 - ADDITIONAL BATCHES (Full mode only - optional, can repeat)
# ============================================================================

if TUNING_MODE == "smoke":
    print("[SMOKE MODE] Skipping additional batches.")
else:
    # ---- Uncomment ONE option below, adjust values, then run this cell ----

    # OPTION (i): Common trials for all models
    # additional_states = manager.continue_optimization(additional_trials=20)

    # OPTION (ii): Per-model trials - adjust counts per model
    # additional_states = manager.continue_optimization(
    #     trials_per_model={
    #         'ctgan': 20,
    #         'tvae': 20,
    #         'copulagan': 20,
    #         'ctabgan': 20,
    #         'ctabganplus': 20,
    #         'ganeraid': 20,
    #         'pategan': 20,
    #         'medgan': 20,
    #     }
    # )

    # OPTION (iii): Time-based budget - minutes per model
    additional_states = manager.continue_optimization(
        time_budget_minutes={
            'ctgan': 30,
            'tvae': 30,
            # 'copulagan': 30,
            # 'ctabgan': 30,
            'ctabganplus': 30,
            # 'ganeraid': 30,
            # 'pategan': 30,
            # 'medgan': 30,
        }
    )

    # print("\nUpdated convergence report:")
    # print(manager.get_diminishing_returns_report().to_string(index=False))

    print("Cell skipped - uncomment an option above to run additional batches")



STAGED OPTIMIZATION - CONTINUATION PHASE
  ctgan: 17 additional trials
  tvae: 82 additional trials
  ctabganplus: 4 additional trials


[CONTINUE] Optimizing CTGAN...
--------------------------------------------------
  Resuming from 30 existing trials


Gen. (-1.58) | Discrim. (-0.17): 100%|██████████| 500/500 [00:34<00:00, 14.52it/s]


[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5598
[OK] TRTS Evaluation: 2 scenarios, Average: 0.6124
[CHART] Combined Score: 0.5808 (Similarity: 0.5598, Accuracy: 0.6124)
[ctgan] Trial 31: Combined Score: 0.5808 (Similarity: 0.5598, Accuracy: 0.6124) Best Score so far: 0.6947


Gen. (-0.51) | Discrim. (0.15): 100%|██████████| 950/950 [01:03<00:00, 14.99it/s] 


[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5344
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9030
[CHART] Combined Score: 0.6818 (Similarity: 0.5344, Accuracy: 0.9030)
[ctgan] Trial 32: Combined Score: 0.6818 (Similarity: 0.5344, Accuracy: 0.9030) Best Score so far: 0.6947


...

Gen. (-0.58) | Discrim. (-0.17): 100%|██████████| 950/950 [01:03<00:00, 14.85it/s]


[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5212
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8887
[CHART] Combined Score: 0.6682 (Similarity: 0.5212, Accuracy: 0.8887)
[ctgan] Trial 43: Combined Score: 0.6682 (Similarity: 0.5212, Accuracy: 0.8887) Best Score so far: 0.6982


Gen. (-1.19) | Discrim. (0.08): 100%|██████████| 950/950 [01:11<00:00, 13.27it/s] 


[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5286
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9167
[CHART] Combined Score: 0.6839 (Similarity: 0.5286, Accuracy: 0.9167)
[ctgan] Trial 44: Combined Score: 0.6839 (Similarity: 0.5286, Accuracy: 0.9167) Best Score so far: 0.6982


Gen. (-1.32) | Discrim. (-0.03):  94%|█████████▎| 936/1000 [01:04<00:04, 13.21it/s]

### 4.6 Save Best Parameters

In [ ]:
# Code Chunk ID: CHUNK_4_6_SAVE
# ============================================================================
# SECTION 4.6 - SAVE BEST PARAMETERS TO CSV
# ============================================================================

from src.utils.parameters import save_best_parameters_to_csv

# Extract studies from manager to globals for saving
for model_name, state in manager.model_states.items():
    if state.study is not None:
        globals()[f"{model_name}_study"] = state.study

# Save all best parameters to CSV for Section 5
save_result = save_best_parameters_to_csv(
    scope=globals(),
    section_number=4,
    dataset_identifier=DATASET_IDENTIFIER
)

print(f"\nParameters saved: {save_result['success']}")
print(f"Files: {save_result.get('files_saved', [])}")

## 5 Final Model Comparison with Best Parameters

### 5.1 Train All Models with Best Parameters from Section 4

In [ ]:
# Code Chunk ID: CHUNK_5_1_BATCH
# ============================================================================
# SECTION 5.1 - BATCH TRAINING WITH BEST PARAMETERS
# ============================================================================

from src.models.batch_training import train_models_batch_with_best_params

# Train all models with best parameters from Section 4 (checkpoint resumes completed models)
section5_results = train_models_batch_with_best_params(
    data=data,
    target_column=target_column,
    models_to_run=models_to_run,
    categorical_columns=categorical_columns,
    n_samples=len(data),
    random_state=42,
    section_number=4,
    dataset_identifier=DATASET_IDENTIFIER,
    scope=globals(),
    verbose=True,
    checkpoint=checkpoint
)

# Show summary of created variables
final_vars = [k for k in globals().keys() if k.endswith('_final')]
print(f"\nFinal synthetic data variables: {sorted(final_vars)}")

### 5.2 Batch Evaluation of Optimized Models

In [ ]:
# Code Chunk ID: CHUNK_5_2_0_A
# ============================================================================
# SECTION 5.2 - OPTIMIZED MODELS BATCH EVALUATION
# ============================================================================

print("SECTION 5.2 - OPTIMIZED MODELS BATCH EVALUATION")
print("=" * 80)

from setup import evaluate_section5_optimized_models

if checkpoint.exists('section_5.2'):
    section5_batch_results = checkpoint.load('section_5.2')['results']
    print("[RESUME] Loaded Section 5.2 evaluation from checkpoint")
    print(f"Models processed: {section5_batch_results['models_processed']}")
    print(f"Results exported to: {section5_batch_results['results_dir']}")
else:
    try:
        section5_batch_results = evaluate_section5_optimized_models(
            section_number=5,
            scope=globals(),
            target_column=TARGET_COLUMN,
            protected_col=NOTEBOOK_CONFIG.get("protected_col"),
            compute_mia=True
        )
        checkpoint.save('section_5.2', {'results': section5_batch_results})
        
        print("\n" + "="*80)
        print("SECTION 5.2 OPTIMIZED MODELS BATCH EVALUATION COMPLETED!")
        print("="*80)
        print(f"Models processed: {section5_batch_results['models_processed']}")
        print(f"Results exported to: {section5_batch_results['results_dir']}")
        
    except Exception as e:
        print(f"Section 5.2 batch evaluation failed: {e}")
        import traceback
        traceback.print_exc()

### 5.3 Final Summary

In [ ]:
# Code Chunk ID: CHUNK_5_3_SUMMARY
# ============================================================================
# SECTION 5.3 - FINAL SUMMARY
# ============================================================================

print("=" * 80)
print("SYNTHETIC DATA GENERATION - FINAL SUMMARY")
print("=" * 80)
print(f"\nDataset: {DATASET_IDENTIFIER}")
print(f"Session: {SESSION_TIMESTAMP}")
print(f"\nResults directories:")
print(f"  Section 2 (EDA): {get_results_path(DATASET_IDENTIFIER, 2)}")
print(f"  Section 3 (Demo): {get_results_path(DATASET_IDENTIFIER, 3)}")
print(f"  Section 4 (HPO): {get_results_path(DATASET_IDENTIFIER, 4)}")
print(f"  Section 5 (Final): {get_results_path(DATASET_IDENTIFIER, 5)}")

# Show staged optimization summary
if 'manager' in dir() and manager is not None:
    print(f"\nStaged Optimization Summary:")
    print("-" * 60)
    summary = manager.get_best_params_summary()
    if len(summary) > 0:
        print(summary[['model', 'best_score', 'total_trials', 'status']].to_string(index=False))

# Show final model performance
if 'section5_results' in dir() and section5_results:
    print(f"\nFinal Model Performance (with optimized parameters):")
    print("-" * 60)
    
    scores = []
    for model_name, result in section5_results.items():
        if result['status'] == 'success':
            score = result.get('objective_score', 0)
            time_taken = result.get('training_time', 0)
            scores.append((model_name, score, time_taken))
    
    # Sort by score descending
    scores.sort(key=lambda x: x[1], reverse=True)
    
    for i, (model, score, time_taken) in enumerate(scores, 1):
        print(f"  {i}. {model.upper()}: score={score:.4f}, time={time_taken:.1f}s")
    
    if scores:
        best_model = scores[0][0]
        print(f"\nBest performing model: {best_model.upper()}")
        print(f"Best synthetic data variable: synthetic_{best_model}_final")

print("\n" + "=" * 80)
print("NOTEBOOK COMPLETE")
print("=" * 80)